In [ ]:
#training process
#!python train.py

In [ ]:
import os
from copy import deepcopy
import torch
import cv2
import numpy as np
import matplotlib.cm as cm
import matplotlib.pyplot as plt
from src.utils.plotting import make_matching_figure
from src.loftr import LoFTR, default_cfg
_default_cfg = deepcopy(default_cfg)
_default_cfg['coarse']['temp_bug_fix'] =True # set to False when using the old ckpt
matcher = LoFTR(config=_default_cfg)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
state_dict_path = "weights.pt"
state_dict = torch.load(state_dict_path, map_location=device)
matcher.load_state_dict(state_dict)
matcher = matcher.eval().to(device)

In [ ]:
import time
# Load example images
img0_pth ='dataset/Left_fig/0000055956.jpg'
img1_pth ='dataset/Right_fig/0000055956.jpg'
img0_raw = cv2.imread(img0_pth, cv2.IMREAD_GRAYSCALE)
img1_raw = cv2.imread(img1_pth, cv2.IMREAD_GRAYSCALE)
assert img0_raw is not None, f"Image at {img0_pth} could not be loaded."
assert img1_raw is not None, f"Image at {img1_pth} could not be loaded."
h, w = img00_raw.shape[:2]
img0_raw = cv2.resize(img0_raw, (480,640))
img1_raw = cv2.resize(img1_raw, (480,640))
img0 = torch.from_numpy(img0_raw).unsqueeze(0).unsqueeze(0).float().to(device) / 255.0
img1 = torch.from_numpy(img1_raw).unsqueeze(0).unsqueeze(0).float().to(device) / 255.0
batch = {'image0': img0, 'image1': img1}
# Inference with LoFTR and get prediction
with torch.no_grad():
    #start_time = time.time()
    matcher(batch)
    #print(matcher(batch))
    mkpts0 = batch['i_ids'].cpu().numpy()
    mkpts1 = batch['j_ids'].cpu().numpy()
    mconf = batch['mconf'].cpu().numpy()
    conf_matrix=batch['conf_matrix'].cpu().numpy()
valid_mask = mconf >= 0.2
mkpts0 = mkpts0[valid_mask]
mkpts1 = mkpts1[valid_mask]
mconf = mconf[valid_mask]

In [ ]:
def draw_features(image, features, img_size, color, draw_text=True):
    indices = range(len(features))
    sx = image.shape[1] / img_size[0]
    sy = image.shape[0] / img_size[1]

    for i, point in zip(indices, features):
        point_int = (int(round(point[0] * sx)), int(round(point[1] * sy)))
        cv2.circle(image, point_int, 2, color, -1, lineType=16)
        if draw_text:
            cv2.putText(image, str(i), point_int, cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1, cv2.LINE_AA)
def get_coarse_match(
    input_height: int,
    input_width: int,
    resolution: int,
    mkpts0: np.ndarray,
    mkpts1: np.ndarray,
    mconf: np.ndarray = None,
):
    """
    Recover coarse-level matches to original image coordinates
    using coarse-cell CENTER (LoFTR-style).

    Args:
        input_height, input_width : original image size
        resolution : coarse resolution (e.g., 32 for LoFTR)
        mkpts0, mkpts1 : coarse grid flatten indices, shape [M]
        mconf : optional confidence, shape [M]

    Returns:
        mkpts0_c : [M, 2] coarse points in image-0 (pixel coords, center of cell)
        mkpts1_c : [M, 2] coarse points in image-1 (pixel coords, center of cell)
        mconf    : [M]
    """

    # -----------------------------
    # 1. image & coarse grid shape
    # -----------------------------
    hw0_i = (input_height, input_width)
    hw0_c = (input_height // resolution, input_width // resolution)
    Hc, Wc = hw0_c

    # -----------------------------
    # 2. safety checks (important)
    # -----------------------------
    mkpts0 = np.asarray(mkpts0)
    mkpts1 = np.asarray(mkpts1)

    assert mkpts0.ndim == 1 and mkpts1.ndim == 1
    assert mkpts0.shape == mkpts1.shape
    assert mkpts0.max() < Hc * Wc
    assert mkpts1.max() < Hc * Wc

    # -----------------------------
    # 3. flatten index -> (row, col)
    # -----------------------------
    cols0 = mkpts0 % Wc
    rows0 = mkpts0 // Wc

    cols1 = mkpts1 % Wc
    rows1 = mkpts1 // Wc

    # -----------------------------
    # 4. center of coarse cell
    #    (LoFTR official behavior)
    # -----------------------------
    scale = resolution

    mkpts0_c = np.stack(
        [(cols0 + 0.5) * scale,
         (rows0 + 0.5) * scale],
        axis=1
    )

    mkpts1_c = np.stack(
        [(cols1 + 0.5) * scale,
         (rows1 + 0.5) * scale],
        axis=1
    )

    # -----------------------------
    # 5. confidence
    # -----------------------------
    if mconf is None:
        mconf = np.ones(len(mkpts0), dtype=np.float32)

    return mkpts0_c, mkpts1_c, mconf
loftr_coarse_resolution=32
mkpts0, mkpts1, mconf = get_coarse_match(640,480, loftr_coarse_resolution,mkpts0,mkpts1)

In [ ]:
x_axis_thre, y_axis_thre = 60,60 # denose in pixel
mkpts0_modify, mkpts1_modify,mconf_modify = mkpts0.copy(), mkpts1.copy(),mconf.copy()
remove_rows = []
for i in range(len(mkpts0_modify)):
    if img0_raw[mkpts0_modify[i].astype(int)[1],mkpts0_modify[i].astype(int)[0]] == 0 or img1_raw[mkpts1_modify[i].astype(int)[1],mkpts1_modify[i].astype(int)[0]] == 0:
        remove_rows.append(i)
if len(remove_rows)>0:
    mkpts0_modify = np.delete(mkpts0_modify, remove_rows, 0)
    mkpts1_modify = np.delete(mkpts1_modify, remove_rows, 0)
    mconf_modify = np.delete(mconf_modify, remove_rows, 0)
x_dif1, y_dif1 = [], []
x_dif2, y_dif2 = [], []
x_dif11, y_dif11 = [], []
for j in range(len(mkpts0_modify)):

    x_dif11.append(mkpts0_modify[j][0]-mkpts1_modify[j][0])
    y_dif11.append(mkpts0_modify[j][1]-mkpts1_modify[j][1])
x_dif=np.array(x_dif11).mean()
y_dif=np.array(y_dif11).mean()

remove_rows = []
for i in range(len(mkpts0_modify)):
    if abs(mkpts0_modify[i][0]-mkpts1_modify[i][0]- x_dif) > x_axis_thre or abs(mkpts0_modify[i][1]-mkpts1_modify[i][1]- y_dif) > y_axis_thre:
        remove_rows.append(i)
        x_dif2.append(x_dif)
        y_dif2.append(y_dif)
    else:
        x_dif2.append(mkpts0_modify[i][0]-mkpts1_modify[i][0])
        x_dif2.append(mkpts0_modify[i][1]-mkpts1_modify[i][1])
if len(remove_rows)>0:
    mkpts0 = np.delete(mkpts0_modify, remove_rows, 0)
    mkpts1 = np.delete(mkpts1_modify, remove_rows, 0)
    mconf = np.delete(mconf_modify, remove_rows, 0)
    

In [ ]:
draw_features(img0_raw, mkpts0,(480,640), color=(0, 0, 255))
draw_features(img1_raw , mkpts1,(480,640), color=(0, 0, 255))
res_img = np.hstack((img0_raw,img1_raw))
plt.figure(figsize=(12, 6)) 
plt.imshow(res_img, cmap='gray')  
plt.axis('off') 
plt.title("Horizontally Stacked Images")
plt.show()
